# Task 1.2 — Key Assumptions
**Paper:** Efficient Additive Kernels via Explicit Feature Maps  
**Student:** Nikhil Raj | Roll No: 230080

---

## Assumption 1

**Assumption:** All input feature values must be strictly non-negative (x ∈ ℝ₊ᴰ, i.e. xᵢ ≥ 0 for all dimensions i).

**Why the method needs it:** The feature map in Equation (19) applies the operations √(xᵢ^γ), cos(L·log(xᵢ)), and sin(L·log(xᵢ)) to each component independently. Both the square root and the logarithm are undefined for non-positive real numbers — log(0) = -∞ and log(x) is complex for x < 0. The kernel itself (e.g. χ²: k(x,y) = 2xy/(x+y)) also becomes undefined or degenerate when x+y = 0. The entire theoretical construction in Section 2 restricts the domain to the non-negative semiaxis ℝ₊, so non-negativity is not just a practical requirement but a foundational mathematical one.

**Violation scenario:** Any standardised or zero-meaned feature representation will violate this assumption. For example, applying sklearn's StandardScaler (which subtracts the mean and divides by standard deviation) to a dataset of continuous measurements like body temperature or stock returns will produce a feature matrix with both positive and negative values. Applying AdditiveChi2Sampler to such data silently produces NaN or -inf values, causing the downstream linear SVM to either crash or train on corrupted features without any warning.

**Paper reference:** Section 2, opening paragraph — "A kernel kₕ: ℝ₊₀ × ℝ₊₀ → ℝ is γ-homogeneous if..." The domain ℝ₊₀ (non-negative semiaxis) is explicitly stated. Also, the feature map Eq. (19) contains √(xᵢ^γ) and log(xᵢ) directly. Section 2 also contains Lemma 2, which discusses the separate (non-trivial) extension needed to handle negative components — confirming that negative support is not automatic.

---

## Assumption 2

**Assumption:** The kernel must be *additive* — it must decompose as a sum of independent scalar kernels applied per-dimension: K(**x**, **y**) = Σᵢ k(xᵢ, yᵢ).

**Why the method needs it:** The efficiency of the entire approach depends on this decomposition. Because the kernel adds over dimensions independently, the feature map can be computed *component-wise* — each input dimension xᵢ is mapped separately using the same scalar function Ψ̂(xᵢ), and the results are concatenated (Equation 13, direct sum ⊕). This makes the transformation O((2n+1)·D) — linear in D. If the kernel had cross-dimensional interactions (e.g. K(**x**,**y**) = (xᵀy)² involves all pairs xᵢyⱼ), the component-wise approach would be invalid, and the Fourier sampling argument in Section 4.2 would break down.

**Violation scenario:** The Gaussian RBF kernel K(**x**,**y**) = exp(-‖**x**-**y**‖²/2σ²) is *not* additive — it computes the Euclidean norm across all dimensions simultaneously. Attempting to apply the homogeneous kernel map component-wise to approximate an RBF kernel would fail. The paper itself acknowledges this and dedicates Section 7 to a separate, more complex construction combining random Fourier features with the homogeneous map specifically to handle this case.

**Paper reference:** Section 2, Equation (7) — "K(**x**,**y**) = Σᴰₗ₌₁ k(xₗ,yₗ)" is the additive combination explicitly defined. Section 3, Equation (13) states that the additive case uses the direct sum ⊕ (stacking), while the multiplicative case uses the Kronecker product ⊗ — and the paper only derives efficient feature maps for the additive case.

---

## Assumption 3

**Assumption:** The kernel must be *γ-homogeneous* — it must satisfy k(cx, cy) = c^γ · k(x, y) for all c ≥ 0 and some fixed γ > 0.

**Why the method needs it:** Homogeneity enables rewriting the kernel in terms of the *ratio* y/x (Equation 2): kₕ(x,y) = (xy)^(γ/2) · K(log(y/x)). This change of variables (from linear space to log-space) converts the 2D kernel function into a 1D signature K(λ) = kₕ(e^(λ/2), e^(-λ/2)), where λ = log(y/x). Bochner's theorem can then be applied to this 1D signature to derive a feature map via Fourier analysis. Without homogeneity, the kernel cannot be factored this way, the signature concept does not apply, and the Fourier derivation in Section 3 is not valid.

**Violation scenario:** The RBF kernel K(**x**,**y**) = exp(-‖**x**-**y**‖²/2σ²) is *stationary* (translation-invariant) but not homogeneous. The paper shows in Section 2 that stationary and homogeneous kernels are related through a log-transform, but requires a separate analysis path. More practically: if a practitioner uses a polynomial kernel K(**x**,**y**) = (xᵀy + c)^d, this is neither purely homogeneous nor stationary in the required sense, and the homogeneous kernel map cannot be applied directly.

**Paper reference:** Section 2, Equation (1) — "A kernel kₕ: ℝ₊₀ × ℝ₊₀ → ℝ is γ-homogeneous if ∀c ≥ 0: kₕ(cx,cy) = c^γ kₕ(x,y)." The entire theoretical development in Sections 2–5 depends on this property. The paper also draws a parallel to *stationary* kernels (Eq. 4–6) to emphasise that homogeneity plays the same structural role for multiplicative scaling as stationarity plays for additive translation.

---

## Assumption 4

**Assumption:** The kernel's spectral density κ(ω) must decay sufficiently fast as |ω| → ∞ for a low approximation order (small n) to be accurate.

**Why the method needs it:** The finite feature map is obtained by truncating the infinite Fourier series at n components (Eq. 21). If κ(ω) decays slowly (heavy-tailed spectrum), then the omitted high-frequency terms contribute significantly to the kernel value, and the approximation will be poor unless n is made very large. The error bound in Lemma 3 (Eq. 25) explicitly depends on how fast κ(ω) decays: for exponentially decaying spectra (like χ², where κ(ω) = sech(πω)) the error decays exponentially in n; for polynomially decaying spectra (like the intersection kernel) the error decays only polynomially, making the intersection kernel harder to approximate at low n.

**Violation scenario:** The intersection kernel (κ(ω) = 2/π · 1/(1+4ω²)) has a spectrum that decays as O(ω⁻²) — a polynomial rate. As confirmed in Section 5, this means significantly more sample_steps are needed to achieve the same approximation quality as χ². For a practitioner using sample_steps=1 (n=1, giving 3 dimensions per feature) on intersection kernel data, the approximation error may be noticeably larger than for χ², leading to degraded classification accuracy compared to the exact kernel.

**Paper reference:** Section 5, Lemma 3 (Eq. 25) — the uniform error bound. Section 5 text — "the approximation error decays as O(exp(-√(min{α/2,γ/4} π(n+1)β))) for the smoother χ² and JS kernels, and as O((n/log n)^(1-β)) for the intersection kernel... the intersection kernel is harder to approximate." Also Figure 1 (bottom panels) shows visually that χ²'s spectrum decays faster than intersection's.
